# Pràctica progressiva — Anàlisi de vendes d'una botiga (Versió Guiada)

**Nivell:** mitjà · **Durada orientativa:** 2–3 sessions de 90 minuts · **Format:** Jupyter Notebook

## Encàrrec professional
Treballes com a tècnic/a d'anàlisi de dades d'una botiga. La persona responsable comercial t'ha lliurat un CSV de vendes i necessita un resum fiable per prendre decisions. Hauràs de preparar les dades, justificar les regles aplicades i comunicar els resultats a un públic no tècnic.

## Objectius
- Llegir CSV amb `csv.DictReader` i explorar-ne l'estructura.
- Validar, normalitzar i convertir dades amb funcions pròpies.
- Separar dades vàlides, corregibles i rebutjades amb traçabilitat.
- Calcular indicadors i fer agrupacions.
- Verificar els resultats i generar fitxers d'informe.
- Formular conclusions prudents i defensar les decisions.

**Fitxers:** posa `vendes_botiga.csv` al mateix directori que aquest notebook. Per al cas final, utilitza `cas_professional_vendes.csv`.

La pràctica es divideix en fases amb un nivell progressiu d'assistència. Utilitza només la biblioteca estàndard de Python.

## Fase 1 — Inspecció guiada

Comença per entendre el fitxer abans de modificar-lo. Cada fila és una línia de venda; els camps són textuals quan es llegeixen del CSV.

In [2]:
import csv
from datetime import datetime

nom_fitxer = "giminasio.csv"


In [11]:

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

print("Camps del CSV:", camps)
print("Registres:", (registres))
print("Nombre total de registres:", len(registres))


Camps del CSV: ['activitat', 'tipus_activitat', 'centre', 'participants', 'durada_min', 'data']
Registres: [{'activitat': 'Accés piscina', 'tipus_activitat': 'Quota', 'centre': 'Girona', 'participants': '20', 'durada_min': '30.1', 'data': '2025-09-16'}, {'activitat': 'Pack de sessions', 'tipus_activitat': 'Entrenament personal', 'centre': 'Terrassa', 'participants': '77', 'durada_min': '124.7', 'data': '2025-11-03'}, {'activitat': 'Accés piscina', 'tipus_activitat': 'Serveis', 'centre': 'Terrassa', 'participants': '28', 'durada_min': '46.2', 'data': '2025-02-09'}, {'activitat': ' Classe ioga  ', 'tipus_activitat': 'Entrenament personal', 'centre': 'Tarragona', 'participants': '64', 'durada_min': '91.5', 'data': '2026-09-16'}, {'activitat': 'Quota mensual', 'tipus_activitat': '   ', 'centre': 'Mataró', 'participants': '2.5', 'durada_min': '5.6', 'data': '2025-04-01'}, {'activitat': 'Classe ioga', 'tipus_activitat': 'Instal·lacions', 'centre': 'Terrassa', 'participants': '28', 'durada_mi

### Exercici 1.1
Calcula quants registres tenen almenys un camp buit o format només per espais. No modifiquis les dades encara.

**Pista:** Pots fer servir `.strip() == ""` sobre cada valor del diccionari per comprovar si un camp està buit o té només espais.

In [26]:
# GUIA PAS A PAS:
# 1. Inicialitza el comptador 'registres_incomplets' a 0.
# 2. Revisa cada registre de la llista 'registres'.
# 3. Comprova si algun dels seus valors està buit després d'aplicar .strip().
# 4. Incrementa el comptador si es detecta algun camp buit.

registres_incomplets = 0

for reg in registres:
    # Comprova si hi ha algun camp buit en la fila
    te_buit = any(val is None or str(val).strip() == "" for val in reg.values())
    if te_buit:
        registres_incomplets += 1

print(f"Registres amb almenys un camp buit/espais: {registres_incomplets}")


Registres amb almenys un camp buit/espais: 15


### Exercici 1.2

**Reflexiona:** quines dades són imprescindibles per calcular l'import d'una venda? Quines són necessàries per segmentar-la/classificar-la?

## Fase 2 — Regles de qualitat i neteja guiada

Aplica inicialment aquestes regles i documenta qualsevol decisió addicional:
1. Elimina espais sobrants dels camps textuals (`strip()`).
2. `unitats` ha de ser un enter estrictament positiu (> 0).
3. `preu_unitari` ha de ser un nombre positiu (> 0); s'accepten coma o punt decimal.
4. La data ha de tenir el format `AAAA-MM-DD` i ser una data vàlida.
5. Producte, categoria i ciutat són camps descriptius obligatoris.
6. Un registre amb valors numèrics o data no interpretables no entra als càlculs. No inventis valors per completar-los.

### Exercici 2 — Funcions de conversió amb gestió d'errors
Implementa funcions de conversió robustes utilitzant blocs `try...except`.

In [30]:
def convertirParticipantsApositivo(valor: str):
    """Retorna un int positiu (>0) o None si no és vàlid."""
    try:
        v = int(str(valor).strip())
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def convertirDurada_minAInt(valor):
    try:
        valor_float = float(str(valor).strip().replace(",", "."))
        if not valor_float > 0:
            return None
        return int(valor_float)
    except (ValueError, TypeError, OverflowError):
        return None

def data_valida(valor: str):
    """Retorna la data si el text correspon a una data vàlida en format AAAA-MM-DD, o None si no és vàlida."""
    try:
        data= datetime.strptime(str(valor).strip(), "%Y-%m-%d")
        return data
    except (ValueError, TypeError):
        return None


### Exercici 3 — Neteja amb traçabilitat

Classifica les dades en tres estructures:
- `vendes_netes`: llista de diccionaris amb dades convertides i camp `import` inclòs.
- `rebutjades`: llista de registres no vàlids.
- `incidencies`: llista de registres/incidències amb el detall per a la traçabilitat (`num_fila`, `tipus`, `camp`, `valor_original`, `accio`).

In [31]:
aceptades= []
incidencies = []
rebutjades = []

obligatoris = ["activitat", "tipus_activitat", "centre"]

for idx, reg in enumerate(registres, start=2): # Comencem a 2 per la capçalera del CSV
    es_valid = True
    reg_net = {}
    
    # 1. Comprovació de camps obligatoris
    for camp in obligatoris:
        val_original = reg.get(camp, "")
        val_net = str(val_original).strip()
        if not val_net:
            es_valid = False
            incidencies.append({
                "fila_csv": idx, "tipus": "REBUTJAT", "camp": camp,
                "valor_original": val_original, "accio": "Camp obligatori buit"
            })
        else:
            if val_original != val_net:
                incidencies.append({
                    "fila_csv": idx, "tipus": "CORRECCIO", "camp": camp,
                    "valor_original": val_original, "accio": "Espais sobrants eliminats"
                })
            reg_net[camp] = val_net
            
    # 2. Conversió deParticipants
    participant_val = convertirParticipantsApositivo(reg.get("participants"))
    if participant_val is None:
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "Participants",
            "valor_original": reg.get("participants"), "accio": "Enter no positiu o no vàlid"
        })
    else:
        reg_net["participants"] = participant_val
        
    # 3. Conversió de durada_min
    durada_min_original = str(reg.get("durada_min", ""))
    durada_min_val = convertirDurada_minAInt(durada_min_original)
    if durada_min_val is None:
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "durada_min",
            "valor_original": durada_min_original, "accio": "Durada no positiva o no numèrica"
        })
    else:
        
        reg_net["durada_min"] = durada_min_val
        
    # 4. Validació de data
    data_orig = str(reg.get("data", "")).strip()
    if not data_valida(data_orig):
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "data",
            "valor_original": data_orig, "accio": "Data amb format incorrecte o no vàlida"
        })
    else:
        reg_net["data"] = data_orig
        
    # Classificació final del registre
    if es_valid:
        aceptades.append(reg_net)
    else:
        rebutjades.append(reg)

print(f"Llegits: {len(registres)}")
print(f"Vàlids: {len(aceptades)}")
print(f"Rebutjats: {len(rebutjades)}")
print(f"Incidències documentades: {len(incidencies)}")


Llegits: 120
Vàlids: 90
Rebutjats: 30
Incidències documentades: 57


## Fase 3 — Indicadors de negoci

Calcula els indicadors globals i realitza les agrupacions per categoria i ciutat.

In [33]:
# GUIA DE CÀLCUL:
# 1. Suma total d'unitats i d'imports a partir de 'vendes_netes'.
# 2. Càlcul de mitjanes: preu unitari mitjà vs import mitjà per línia.
# 3. Agrupació en diccionaris acumulant la facturació per clau.

# total_participants = sum(v["participants"] for v in aceptades)

# resum_ciutat = {}
# resum_tipoActividad = {}

# tipus_diferents = set(
#     v["tipus_activitat"] for v in aceptades
# )

# for tip in tipus_diferents:
#     resum_tipoActividad[tip] = sum(
#         v["participants"] for v in aceptades if v["tipus_activitat"] == tip)

# ciutats_diferents = set(
#     v["centre"] for v in aceptades
# )

# for ciu in ciutats_diferents:
#     resum_ciutat[ciu] = sum(
#         v["participants"]
#         for v in aceptades
#         if v["centre"] == ciu
#     )

# print(f"Total participants: {total_participants} personas")
# print(f"totalpor tipo de actividad: {resum_tipoActividad} personas")
# print(f"totalpor ciutat: {resum_ciutat} personas")
# ==========================================
# GUIA DE CÀLCUL (Adaptada al sector del Gimnàs)
# ==========================================

# 1. Suma total de participants i durades a partir de 'aceptades' (registres netejats)
total_participants = sum(r["participants"] for r in aceptades)
durada_total_global = sum(r["durada_min"] for r in aceptades)

print(f"1. Suma total de participants: {total_participants}")
print(f"   Suma total de durades: {durada_total_global:.2f} min")


# 2. Càlcul de mitjanes: durada mitjana per activitat vs participants mitjans per línia
num_registres_valids = len(aceptades)

durada_mitjana = durada_total_global / num_registres_valids if num_registres_valids > 0 else 0
participants_mitjans_linia = total_participants / num_registres_valids if num_registres_valids > 0 else 0

print(f"\n2. Càlcul de mitjanes:")
print(f"   - Durada mitjana per activitat: {durada_mitjana:.2f} min")
print(f"   - Participants mitjans per línia: {participants_mitjans_linia:.2f}")


# 3. Agrupació en diccionaris acumulant per clau (Tipus d'activitat i Centre)
resum_categoria = {}
resum_centre = {}

for r in aceptades:
    # Agrupació per tipus d'activitat (categoria)
    cat = r["tipus_activitat"].strip()
    resum_categoria[cat] = resum_categoria.get(cat, 0) + r["participants"]
    
    # Agrupació per centre
    c = r["centre"].strip()
    if not c:
        c = "Desconegut"
    resum_centre[c] = resum_centre.get(c, 0) + r["participants"]

print(f"\n3. Agrupació en diccionaris:")
print(f"   - Desglossament per Categoria: {resum_categoria}")
print(f"   - Desglossament per Centre: {resum_centre}")







1. Suma total de participants: 3534
   Suma total de durades: 9601.00 min

2. Càlcul de mitjanes:
   - Durada mitjana per activitat: 106.68 min
   - Participants mitjans per línia: 39.27

3. Agrupació en diccionaris:
   - Desglossament per Categoria: {'Quota': 483, 'Entrenament personal': 908, 'Serveis': 734, 'Instal·lacions': 782, 'Classe dirigida': 627}
   - Desglossament per Centre: {'Girona': 779, 'Terrassa': 910, 'Tarragona': 486, 'Barcelona': 793, 'Mataró': 566}


### Control de qualitat
Validació automàtica de la consistència de les dades mitjançant assercions (`assert`).

In [ ]:
# 1. Comprovació de la quantitat de registres
# assert len(aceptades) + len(rebutjades) == len(registres), "Error: La suma de aceptades i rebutjats no quadrat amb el total."
# # 2. Comprovació de valors positius

# assert all(
#     v["participants"] > 0 and v["durada_min"] > 0
#     for v in aceptades
# ), "Error: Hi ha participants o durades no positives."

# # 3. Comprovació dels camps obligatoris
# assert all(
#     str(v[camp]).strip() != ""
#     for v in aceptades
#     for camp in ["activitat", "tipus_activitat", "centre"]
# ), "Error: Hi ha camps obligatoris buits."



# print("Tots els controls de qualitat s'han superat amb èxit!")
# ==========================================
# FASE 6: CONTROLS DE QUALITAT (ASSERTIONS)
# ==========================================


# 1. Comprovació de la quantitat de registres
assert len(aceptades) + len(rebutjades) == len(registres), "Error: La suma de aceptades i rebutjats no quadrat amb el total."

# 2. Comprovació de valors positius
assert all(
    v["participants"] > 0 and v["durada_min"] > 0
    for v in aceptades
), "Error: Hi ha participants o durades no positives."

# 3. Comprovació dels camps obligatoris principals
assert all(
    str(v[camp]).strip() != ""
    for v in aceptades
    for camp in ["activitat", "tipus_activitat", "centre"]
), "Error: Hi ha camps obligatoris buits."

print("Tots els controls de qualitat s'han superat amb èxit!")

Tots els controls de qualitat s'han superat amb èxit!


## Fase 4 — Interpretació i comunicació

Respon a les preguntes analítiques identificant els màxims i avaluant les conclusions de negoci.

In [ ]:
# Trobar la tipus d'activitat i ciutat amb més facturació
# Trobar el tipus d'activitat i centre amb més participants
top_tipo_activitat = max(resum_categoria.items(), key=lambda x: x[1]) if resum_categoria else (None, 0)
top_centre = max(resum_centre.items(), key=lambda x: x[1]) if resum_centre else (None, 0)

print(f"1. Tipus d'activitat amb més participants: {top_tipo_activitat[0]} ({top_tipo_activitat[1]} participants)")
print(f"   Centre amb més participants: {top_centre[0]} ({top_centre[1]} participants)")


1. Tipus d'activitat amb més participants: Entrenament personal (908 participants)
   Centre amb més participants: Terrassa (910 participants)


## Fase 5 — Lliurament professional

Generació dels tres fitxers de sortida: `informe_vendes.txt`, `incidencies.csv` i `vendes_netes.csv`.

In [ ]:
import csv
from datetime import datetime

# Nom del fitxer assignat
nom_fitxer_csv = "giminasio.csv"



nom_informe = "informe_resultats.txt"
nom_incidencies = "incidencies.csv"
nom_netes = "registres_nets.csv"

if aceptades:
    with open(nom_netes, mode="w", encoding="utf-8-sig", newline="") as f:
        escriptor = csv.DictWriter(f, fieldnames=aceptades[0].keys())
        escriptor.writeheader()
        escriptor.writerows(aceptades)

with open(nom_incidencies, mode="w", encoding="utf-8-sig", newline="") as f:
    camps_inc = ["fila_csv", "tipus", "camp", "valor_original", "accio"]
    escriptor = csv.DictWriter(f, fieldnames=camps_inc)
    escriptor.writeheader()
    escriptor.writerows(incidencies)


Fitxers generats correctament amb utf-8-sig!


Fase 6:Resultats i conclusions

In [35]:
text_informe = f"""=========================================
INFORME EXECUTIU - GIMNÀS I FITNESS (AC4)
=========================================
Data de generació: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Cobertura: {len(aceptades)} / {len(registres)} registres vàlids ({len(rebutjades)} rebutjats)

INDICADORS GLOBALS:
- Total registres inicials: {len(registres)}
- Total registres vàlids: {len(aceptades)}
- Total registres rebutjats: {len(rebutjades)}
- Suma total participants: {total_participants}
- Durada mitjana per activitat: {durada_mitjana:.2f} min

DESGLOSSAMENT PER TIPUS D'ACTIVITAT:
"""
for cat, val in resum_categoria.items():
    text_informe += f"- {cat}: {val}\n"

text_informe += "\nDESGLOSSAMENT PER CENTRE:\n"
for centre, val in resum_centre.items():
    text_informe += f"- {centre}: {val}\n"

with open(nom_informe, mode="w", encoding="utf-8-sig") as f:
    f.write(text_informe)

print("Fitxers generats correctament amb utf-8-sig!")

Fitxers generats correctament amb utf-8-sig!
